The goal of this code is to calculate the image metrics for models trained on PCCT and Conventional CT data.

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive/')

Mounted at /content/gdrive/


In [ ]:
!pip install pytorch-msssim

In [ ]:
import torch
from torch import nn, optim
from torch.utils.data import Dataset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import os
import numpy as np
from skimage.metrics import structural_similarity as skimage_ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
import matplotlib.gridspec as gridspec

In [ ]:
class makeDatasetPCCT(Dataset):
    def __init__(self,data_dir,transform=None,mode="train"):
        self.transform = transform
        self.data_dir_mode = os.path.join(data_dir,mode)
        self.mode = mode

    def __len__(self):
        if self.mode=="train":
          return TRAIN_VAL # the number of training images
        elif self.mode=="test":
          return TEST_VAL # the number of test images
        else:
          return VALIDATION_VAL # the number of validation images

    def __getitem__(self, idx):
        reduced_sinogram_path = os.path.join(self.data_dir_mode,REDUCED_DOSE, f"{idx}.npy")
        reduced_sinogram = torch.from_numpy(np.load(reduced_sinogram_path)).float()


        full_sinogram_path = os.path.join(self.data_dir_mode,f"{FULL_DOSE}", f"{idx}.npy")
        full_sinogram = torch.from_numpy(np.load(full_sinogram_path)).float().unsqueeze(0)

        if self.transform:
            reduced_sinogram = self.transform(reduced_sinogram)
            full_sinogram = self.transform(full_sinogram)
        return reduced_sinogram, full_sinogram


class makeDatasetConventional(Dataset):
    def __init__(self,data_dir,transform=None,mode="train"):
        self.transform = transform
        self.data_dir_mode = os.path.join(data_dir,mode)
        self.mode = mode

    def __len__(self):
        if self.mode=="train":
          return TRAIN_VAL # the number of training images
        elif self.mode=="test":
          return TEST_VAL # the number of test images
        else:
          return VALIDATION_VAL # the number of validation images

    def __getitem__(self, idx):
        reduced_sinogram_path = os.path.join(self.data_dir_mode,REDUCED_DOSE, f"{idx}.npy")
        reduced_sinogram = torch.from_numpy(np.load(reduced_sinogram_path)).float().unsqueeze(0)


        full_sinogram_path = os.path.join(self.data_dir_mode,f"{FULL_DOSE}", f"{idx}.npy")
        full_sinogram = torch.from_numpy(np.load(full_sinogram_path)).float().unsqueeze(0)

        if self.transform:
            reduced_sinogram = self.transform(reduced_sinogram)
            full_sinogram = self.transform(full_sinogram)
        return reduced_sinogram, full_sinogram


In [ ]:
# model architecture

class UNet(nn.Module):
    def __init__(self, channels,relu_slope, input_channel):
        super(UNet, self).__init__()

        self.initial_conv = nn.Sequential(
            nn.Conv2d(input_channel, channels, kernel_size=3, padding=1),
            nn.LeakyReLU(relu_slope),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.LeakyReLU(relu_slope),
        )

        # encoder 1
        self.enc1 = nn.Sequential(
            nn.Conv2d(channels, channels*2, kernel_size=3, stride=2, padding=1),
            nn.LeakyReLU(relu_slope),
            nn.Conv2d(channels*2, channels*2, kernel_size=3, padding=1),
            nn.LeakyReLU(relu_slope)
        )

        # encoder 2
        self.enc2 = nn.Sequential(
            nn.Conv2d(channels*2, channels*4, kernel_size=3, stride=2, padding=1),
            nn.LeakyReLU(relu_slope),
            nn.Conv2d(channels*4, channels*4, kernel_size=3, padding=1),
            nn.LeakyReLU(relu_slope)
        )

        # encoder 3
        self.enc3 = nn.Sequential(
            nn.Conv2d(channels*4, channels*8, kernel_size=3, stride=2, padding=1),
            nn.LeakyReLU(relu_slope),
            nn.Conv2d(channels*8, channels*8, kernel_size=3, padding=1),
            nn.LeakyReLU(relu_slope)
        )

        # encoder 4
        self.enc4 = nn.Sequential(
            nn.Conv2d(channels*8, channels*16, kernel_size=3, stride=2, padding=1),
            nn.LeakyReLU(relu_slope),
            nn.Conv2d(channels*16, channels*16, kernel_size=3, padding=1),
            nn.LeakyReLU(relu_slope)
        )

        # bottleneck
        self.bottleneck = nn.Sequential(
            nn.Conv2d(channels*16, channels*32, kernel_size=3, padding=1),
            nn.LeakyReLU(relu_slope),
            nn.Conv2d(channels*32, channels*32, kernel_size=3, padding=1),
            nn.LeakyReLU(relu_slope)
        )

        # decoder and upsample 4
        self.up4 = nn.Sequential(
            nn.ConvTranspose2d(channels*32, channels*16, kernel_size=2, stride=2),
            nn.LeakyReLU(relu_slope)
        )
        self.dec4 = nn.Sequential(
            nn.Conv2d(channels*16 + channels*8, channels*16, kernel_size=3, padding=1),
            nn.LeakyReLU(relu_slope),
            nn.Conv2d(channels*16, channels*16, kernel_size=3, padding=1),
            nn.LeakyReLU(relu_slope)
        )

        # decoder and upsample 3
        self.up3 = nn.Sequential(
            nn.ConvTranspose2d(channels*16, channels*8, kernel_size=2, stride=2),
            nn.LeakyReLU(relu_slope)
        )
        self.dec3 = nn.Sequential(
            nn.Conv2d(channels*8 + channels*4, channels*8, kernel_size=3, padding=1),
            nn.LeakyReLU(relu_slope),
            nn.Conv2d(channels*8, channels*8, kernel_size=3, padding=1),
            nn.LeakyReLU(relu_slope)
        )

        # decoder and upsample 2
        self.up2 = nn.Sequential(
            nn.ConvTranspose2d(channels*8, channels*4, kernel_size=2, stride=2),
            nn.LeakyReLU(relu_slope)
        )
        self.dec2 = nn.Sequential(
            nn.Conv2d(channels*4 + channels*2, channels*4, kernel_size=3, padding=1),
            nn.LeakyReLU(relu_slope),
            nn.Conv2d(channels*4, channels*4, kernel_size=3, padding=1),
            nn.LeakyReLU(relu_slope)
        )

        # decoder and upsample 1
        self.up1 = nn.Sequential(
            nn.ConvTranspose2d(channels*4, channels*2, kernel_size=2, stride=2),
            nn.LeakyReLU(relu_slope)
        )
        self.dec1 = nn.Sequential(
            nn.Conv2d(channels*2 + channels, channels*2, kernel_size=3, padding=1),
            nn.LeakyReLU(relu_slope),
            nn.Conv2d(channels*2, channels*2, kernel_size=3, padding=1),
            nn.LeakyReLU(relu_slope)
        )

        # final decoder
        self.dec0 = nn.Sequential(
            nn.Conv2d(channels*2, channels, kernel_size=3, padding=1),
            nn.LeakyReLU(relu_slope)
        )

        # final convolution
        self.out_conv = nn.Conv2d(channels, 1, kernel_size=1)

    def forward(self, x):
        e0 = self.initial_conv(x)
        e1 = self.enc1(e0)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)

        # bottleneck
        b = self.bottleneck(e4)

        d4 = self.dec4(torch.cat([self.up4(b), e3], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e2], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e1], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e0], dim=1))
        d0 = self.dec0(d1)

        # output
        out = self.out_conv(d0)
        return out


In [ ]:
# defining the combined loss function
class CombinedLoss(nn.Module):
    def __init__(self, alpha=0.85):
        super(CombinedLoss, self).__init__()
        self.alpha = alpha
        self.l1_loss = nn.L1Loss()

    def forward(self, prediction, target):
        l1 = self.l1_loss(prediction, target)
        ssim_val = ssim(prediction, target, data_range=1.0, size_average=True)
        combined = self.alpha * (1 - ssim_val) + (1 - self.alpha) * l1

        return combined

In [ ]:
l1_loss_func = nn.L1Loss()
loss_function = CombinedLoss(alpha=0.85)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# The values for the number of images in train, test, and validation as well as the number of projections for the full-dose
TRAIN_VAL = 648
TEST_VAL = 144
VALIDATION_VAL = 72
FULL_DOSE = 3600

In [ ]:
# the hyperparameters used during training
batch_sizeA = 16
channelsA = 16
relu_slopeA = 0.01
input_channelA = 6
inputA = "PCCT" # or "CT"
typeA = "PigTail" # or "ChickenDrumstick"
REDUCED_DOSE = "900" # the number of projections of the reduced dose
RUN_NUM = 3 # which run it was

modelA = UNet(channelsA, relu_slopeA, input_channelA)
model_pathA = f"/content/gdrive/MyDrive/Honours_Thesis/2_PIGTAIL_MODELS/UNET/run1/best_model_Combined_Loss_lr:0.0003_batch:16_epoch:98.pth" # path of model to test
modelA.load_state_dict(torch.load(model_pathA, weights_only=True))
modelA.to(device)
modelA.eval()

In [ ]:
# the defined data transform
dataTransforms=transforms.Compose([
    transforms.Resize((512,512)),
])

# making the dataset depending if it is conventional CT or PCCT
if (inputA == "PCCT"):
  datasetA = makeDatasetPCCT(data_dir=f"/content/gdrive/MyDrive/Honours_Thesis/1.2_DATA/{typeA}/NOT_OVERLAPPING/{FULL_DOSE}_to_{REDUCED_DOSE}",mode="test")
else:
  datasetA = makeDatasetConventional(data_dir=f"/content/gdrive/MyDrive/Honours_Thesis/1.2_DATA/{typeA}/Conventional_NOT_OVERLAPPING/{FULL_DOSE}_to_{REDUCED_DOSE}",transform=dataTransforms, mode="test")
loaderA = torch.utils.data.DataLoader(datasetA, batch_size=batch_sizeA)

In [ ]:
# the metrics to record for each image
ssim_valsA = []
psnr_valsA = []
l1_valsA = []
loss_valsA = []
gt_imagesA = []
total_imagesA = 0
bin_titles = ["25-33 keV", "33-41 keV", "41-50 keV", "50-59 keV", "59-68 keV", ">68 keV"] # the energy bins

if (inputA!="PCCT"):
  for data in loaderA:
    reduced_sinograms, full_sinograms = data
    reconstructed = modelA(reduced_sinograms.float().to(device))
    for i in range(len(reconstructed)):
      ground_truth_img = full_sinograms[i].cpu().detach().numpy()[0,:,:]
      reduced_img = reduced_sinograms[i].cpu().detach().numpy()[0,:,:]
      reconstructed_img = reconstructed[i].cpu().detach().numpy()[0,:,:]

      # calculating the metrics
      psnr_val_output = psnr(ground_truth_img, reconstructed_img,data_range=1.0)
      ssim_val_output = skimage_ssim(ground_truth_img,reconstructed_img, data_range=1.0)
      reduced_img_tensor = torch.from_numpy(reduced_img).float().to(device)
      reconstructed_img_tensor = torch.from_numpy(reconstructed_img).float().to(device).unsqueeze(0).unsqueeze(0)
      ground_truth_img_tensor = torch.from_numpy(ground_truth_img).float().to(device).unsqueeze(0).unsqueeze(0)
      l1_output = l1_loss_func(reconstructed_img_tensor, ground_truth_img_tensor)
      loss_output = loss_function(reconstructed_img_tensor, ground_truth_img_tensor)

      total_imagesA += 1
      ssim_valsA.append(ssim_val_output)
      psnr_valsA.append(psnr_val_output)
      l1_valsA.append(l1_output)
      loss_valsA.append(loss_output)
      gt_imagesA.append(np.sum(ground_truth_img))

      # plotting the output from the model for every 10th image
      if (total_imagesA%10==0):
        fig, axes = plt.subplots(1, 3, figsize=(20, 7))

        images = [reduced_img, ground_truth_img, reconstructed_img]
        titles = ['Input', 'Ground Truth', 'Output']

        for i, ax in enumerate(axes):
            im = ax.imshow(images[i], cmap='gray', vmin=0.00, vmax=0.07)
            ax.set_title(titles[i], fontsize=22, pad=15)
            ax.axis('off')

        fig.subplots_adjust(right=0.88, wspace=0.1)

        cbar_ax = fig.add_axes([0.90, 0.2, 0.015, 0.6])
        cbar = fig.colorbar(im, cax=cbar_ax)
        cbar.set_label(r'$mm^{-1}$', rotation=270, labelpad=25, size=25)
        cbar.ax.tick_params(labelsize=15)

        plt.savefig(f'Combined_Results_{total_imagesA}.pdf', format='pdf', dpi=300, bbox_inches='tight')
        plt.show()


        # zoomed-in window
        y_min, y_max = 150, 300
        x_min, x_max = 225, 375

        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        images = [reduced_img, ground_truth_img, reconstructed_img]
        titles = ['Input (Zoomed)', 'Ground Truth (Zoomed)', 'Output (Zoomed)']

        for i, ax in enumerate(axes):
            im = ax.imshow(images[i], cmap='gray', vmin=0.015, vmax=0.055)

            ax.set_ylim(y_max, y_min) # image origin is top-left
            ax.set_xlim(x_min, x_max)

            ax.set_title(titles[i], fontsize=20)
            ax.axis('off')

        fig.subplots_adjust(right=0.88)

        cbar_ax = fig.add_axes([0.90, 0.15, 0.015, 0.7])
        cbar = fig.colorbar(im, cax=cbar_ax)
        cbar.set_label(r'$mm^{-1}$', rotation=270, labelpad=25, size=25)
        cbar.ax.tick_params(labelsize=15)

        plt.savefig(f'Zoomed_Combined_Results_{total_imagesA}.pdf', format='pdf', dpi=300, bbox_inches='tight')
        plt.show()
        print(f"END OF IMAGE {total_imagesA}")

else:
  for data in loaderA:
    reduced_sinograms, full_sinograms = data
    reconstructed = modelA(reduced_sinograms.float().to(device))
    for i in range(len(reconstructed)):
      ground_truth_img = np.sum(full_sinograms[i].cpu().detach().numpy(), axis=0)
      reduced_img = reduced_sinograms[i].cpu().detach().numpy()
      reconstructed_img = reconstructed[i].cpu().detach().numpy()[0,:,:]

      # calculating the metrics
      psnr_val_output = psnr(ground_truth_img, reconstructed_img,data_range=1.0)
      ssim_val_output = skimage_ssim(ground_truth_img,reconstructed_img, data_range=1.0)
      reconstructed_img_tensor = torch.from_numpy(reconstructed_img).float().to(device).unsqueeze(0).unsqueeze(0)
      ground_truth_img_tensor = torch.from_numpy(ground_truth_img).float().to(device).unsqueeze(0).unsqueeze(0)
      l1_output = l1_loss_func(reconstructed_img_tensor, ground_truth_img_tensor)
      loss_output = loss_function(reconstructed_img_tensor, ground_truth_img_tensor)

      total_imagesA += 1
      ssim_valsA.append(ssim_val_output)
      psnr_valsA.append(psnr_val_output)
      l1_valsA.append(l1_output)
      loss_valsA.append(loss_output)
      gt_imagesA.append(ground_truth_img)

      # plotting the output from the model for every 10th image
      if (total_imagesA % 10 == 0):
        fig = plt.figure(figsize=(22, 10))
        gs = gridspec.GridSpec(3, 4, figure=fig, width_ratios=[1, 1, 2.2, 2.2], hspace=0.3, wspace=0.1)

        for i in range(6):
            row = i // 2
            col = i % 2
            ax = fig.add_subplot(gs[row, col])
            im_input = ax.imshow(reduced_img[i, :, :], cmap='gray', vmin=0.00, vmax=0.07)
            ax.set_title(bin_titles[i], fontsize=14, pad=8)
            ax.axis('off')

        ax_input_title = fig.add_subplot(gs[0, 0:2])
        ax_input_title.set_title('Input', fontsize=25, pad=45)
        ax_input_title.axis('off')

        ax_gt = fig.add_subplot(gs[:, 2])
        ax_gt.imshow(ground_truth_img, cmap='gray', vmin=0.00, vmax=0.07)
        ax_gt.set_title('Ground Truth', fontsize=25, pad=15)
        ax_gt.axis('off')

        ax_out = fig.add_subplot(gs[:, 3])
        im = ax_out.imshow(reconstructed_img, cmap='gray', vmin=0.00, vmax=0.07)
        ax_out.set_title('Output', fontsize=25, pad=15)
        ax_out.axis('off')

        fig.subplots_adjust(right=0.88)
        cbar_ax = fig.add_axes([0.91, 0.2, 0.015, 0.6])
        cbar = fig.colorbar(im, cax=cbar_ax)
        cbar.set_label(r'$mm^{-1}$', rotation=270, labelpad=30, size=25)
        cbar.ax.tick_params(labelsize=15)

        plt.savefig(f'Combined_Results_{total_imagesA}_{REDUCED_DOSE}{inputA}.pdf', format='pdf', dpi=300, bbox_inches='tight')
        plt.show()

        # zoomed-in window
        y_min, y_max = 150, 300
        x_min, x_max = 250, 400

        fig = plt.figure(figsize=(22, 10))

        gs = gridspec.GridSpec(3, 4, figure=fig, width_ratios=[1, 1, 2.2, 2.2], hspace=0.3, wspace=0.1)

        for i in range(6):
            row = i // 2
            col = i % 2
            ax = fig.add_subplot(gs[row, col])
            im_input = ax.imshow(reduced_img[i, :, :], cmap='gray', vmin=0.015, vmax=0.055)

            ax.set_ylim(y_max, y_min)
            ax.set_xlim(x_min, x_max)

            ax.set_title(f'{bin_titles[i]}', fontsize=14, pad=8)
            ax.axis('off')

        ax_input_title = fig.add_subplot(gs[0, 0:2])
        ax_input_title.set_title('Input (Zoomed)', fontsize=25, pad=45)
        ax_input_title.axis('off')

        ax_gt = fig.add_subplot(gs[:, 2])
        ax_gt.imshow(ground_truth_img, cmap='gray', vmin=0.015, vmax=0.055)
        ax_gt.set_ylim(y_max, y_min)
        ax_gt.set_xlim(x_min, x_max)
        ax_gt.set_title('Ground Truth (Zoomed)', fontsize=25, pad=15)
        ax_gt.axis('off')

        ax_out = fig.add_subplot(gs[:, 3])
        im = ax_out.imshow(reconstructed_img, cmap='gray', vmin=0.015, vmax=0.055)
        ax_out.set_ylim(y_max, y_min)
        ax_out.set_xlim(x_min, x_max)
        ax_out.set_title('Output (Zoomed)', fontsize=25, pad=15)
        ax_out.axis('off')

        fig.subplots_adjust(right=0.88)
        cbar_ax = fig.add_axes([0.91, 0.2, 0.015, 0.6])
        cbar = fig.colorbar(im, cax=cbar_ax)
        cbar.set_label(r'$mm^{-1}$', rotation=270, labelpad=30, size=25)
        cbar.ax.tick_params(labelsize=15)

        plt.savefig(f'Zoomed_Combined_Results_{total_imagesA}_{REDUCED_DOSE}{inputA}.pdf', format='pdf', dpi=300, bbox_inches='tight')
        plt.show()
        print(f"END OF IMAGE {total_imagesA}")

imageA value for comparisons:

450PCCT v.s. 450Conventional => 30
        y_min, y_max = 150, 300
        x_min, x_max = 150, 300


450PCCT v.s. 900PCCT => 130
        y_min, y_max = 250, 400
        x_min, x_max = 250, 400


900PCCT v.s. 900Conventional => 70
        y_min, y_max = 150, 300
        x_min, x_max = 250, 400


90PCCT v.s. 90Conventional => 220
        y_min, y_max = 150, 300
        x_min, x_max = 250, 400

90PCCT v.s. 45PCCT => 270
        y_min, y_max = 150, 300
        x_min, x_max = 250, 400

45PCCT v.s. 45Conventional => 15
        y_min, y_max = 150, 300
        x_min, x_max = 225, 375